In [2]:
%pip install scikit-learn torch torchvision transformers tqdm numpy matplotlib opencv-python --quiet

You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from transformers import ViTImageProcessor, ViTForImageClassification
from tqdm import tqdm
import sklearn


/Users/kranthi/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/kranthi/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset_path = "VR/Data"


In [5]:
# Define data augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),  
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])


image_paths, labels = [], []
room_labels = sorted(os.listdir(dataset_path))
room_labels = [label for label in room_labels if label != ".DS_Store"]
label_to_idx = {label: idx for idx, label in enumerate(room_labels)}


for room in room_labels:
    if room == ".DS_Store":
        continue
    room_path = os.path.join(dataset_path, room)
    if os.path.isdir(room_path):
        for img_file in os.listdir(room_path):
            if img_file.lower().endswith((".png", ".jpg", ".jpeg")):
                image_paths.append(os.path.join(room_path, img_file))
                labels.append(label_to_idx[room])

# Convert to NumPy and split into train/test
from sklearn.model_selection import train_test_split
image_paths, labels = np.array(image_paths), np.array(labels)
train_paths, test_paths, train_labels, test_labels = train_test_split(image_paths, labels, test_size=0.2, random_state=42)

print(f"Training: {len(train_paths)}, Testing: {len(test_paths)}")

Training: 4707, Testing: 1177


In [6]:
class RoomDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(self.labels[idx])

# Load datasets
train_dataset = RoomDataset(train_paths, train_labels, transform=train_transform)
test_dataset = RoomDataset(test_paths, test_labels, transform=test_transform)

# DataLoader with mini-batches
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


In [7]:
device = torch.device("mps") if torch.backends.mps.is_available() else "cpu"

# Load ViT Model
image_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=len(room_labels),
    ignore_mismatched_sizes=True
).to(device)

# Fine-tune ALL layers (instead of freezing backbone)
for param in model.parameters():
    param.requires_grad = True

# Print a message to indicate that the model has been loaded and is ready for training
print("Model loaded and ready for training.")
print(device)


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([21]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([21, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded and ready for training.
mps


In [8]:
# Compute class weights (for class imbalance)
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(train_labels), y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# Define loss function with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer & Scheduler
optimizer = optim.AdamW(model.parameters(), lr=3e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)


In [9]:
print(len(room_labels))

21


In [10]:
def train_vit_finetune(num_epochs=30, freeze_epochs=10):
    best_acc = 0

    for epoch in range(num_epochs):
        # Step 1: Unfreeze after N epochs
        if epoch == freeze_epochs:
            print(f"🔓 Unfreezing ViT backbone at epoch {epoch}")
            for param in model.vit.parameters():
                param.requires_grad = True

        model.train()
        total_loss, correct_predictions, total_samples = 0, 0, 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

        for batch in progress_bar:
            inputs, labels = batch
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs).logits
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * inputs.size(0)
            predicted_labels = torch.argmax(outputs, dim=1)
            correct_predictions += (predicted_labels == labels).sum().item()
            total_samples += labels.size(0)

            progress_bar.set_postfix({
                "Loss": loss.item(),
                "Accuracy": f"{(correct_predictions / total_samples) * 100:.2f}%"
            })

        epoch_accuracy = (correct_predictions / total_samples) * 100
        print(f"Epoch {epoch+1} - Loss: {total_loss/total_samples:.4f} - Accuracy: {epoch_accuracy:.2f}%")
        scheduler.step()

        if epoch_accuracy > best_acc:
            best_acc = epoch_accuracy
            torch.save(model.state_dict(), "best_vit_model.pth")
            print("✅ Saved new best model")

# Freeze backbone initially
for param in model.vit.parameters():
    param.requires_grad = False

# Lower LR for fine-tuning
optimizer = optim.AdamW(model.parameters(), lr=2e-4 )

# ✅ Run the updated training loop
train_vit_finetune(num_epochs=30, freeze_epochs=10)


Epoch 1/30:   1%|          | 1/148 [00:01<03:43,  1.52s/it, Loss=3.28, Accuracy=9.38%]

Epoch 1/30: 100%|██████████| 148/148 [02:54<00:00,  1.18s/it, Loss=1.88, Accuracy=40.68%]
/Users/kranthi/Library/Python/3.9/lib/python/site-packages/torch/optim/lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Epoch 1 - Loss: 2.4341 - Accuracy: 40.68%
✅ Saved new best model


Epoch 2/30: 100%|██████████| 148/148 [02:47<00:00,  1.13s/it, Loss=0.836, Accuracy=68.96%]


Epoch 2 - Loss: 1.5233 - Accuracy: 68.96%
✅ Saved new best model


...

Epoch 15 - Loss: 0.0744 - Accuracy: 98.07%
✅ Saved new best model


Epoch 16/30: 100%|██████████| 148/148 [04:57<00:00,  2.01s/it, Loss=0.000143, Accuracy=98.96%]


Epoch 16 - Loss: 0.0551 - Accuracy: 98.96%
✅ Saved new best model


Epoch 17/30:  57%|█████▋    | 84/148 [13:11<10:03,  9.42s/it, Loss=0.204, Accuracy=96.24%]    


KeyboardInterrupt: 

In [11]:
def evaluate_vit():
    model.eval()
    correct_predictions, total_samples = 0, 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs).logits
            predicted_labels = torch.argmax(outputs, dim=1)
            correct_predictions += (predicted_labels == labels).sum().item()
            total_samples += labels.size(0)

    accuracy = (correct_predictions / total_samples) * 100
    print(f"Test Accuracy: {accuracy:.2f}%")

evaluate_vit()


Test Accuracy: 89.38%


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install seaborn
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import torch


# Step 1: Collect true & predicted labels from validation set
y_true = []
y_pred = []

model.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        outputs = model(imgs).logits
        preds = torch.argmax(outputs, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# Step 2: Generate confusion matrix
room_names = list(room_labels)  # le is your LabelEncoder
cm = confusion_matrix(y_true, y_pred)
df_cm = pd.DataFrame(cm, index=room_names, columns=room_names)

# Step 3: Plot the heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(df_cm, annot=True, fmt='d', cmap="Blues", cbar=False)
plt.title("🧭 Confusion Matrix: Predicted vs Actual Room Labels")
plt.xlabel("Predicted Room")
plt.ylabel("Actual Room")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


NameError: name 'model' is not defined

In [13]:
# torch.save(model.state_dict(), "vit_model.pth")  # Optional redundant save
# with open('room_labels.pkl', 'wb') as f:
#     pickle.dump(room_labels, f)
